# PySpark Cluster-Mode Test Sweep

Comprehensive RDD + DataFrame coverage against a real cluster (`spark://...`).
Run top to bottom. Each section is self-contained. Note any cell that errors —
we expect **everything here to pass** on PySpark; this notebook is the baseline
we compare the Scala notebook against.

## 0. Sanity check — confirm cluster, not local

In [ ]:
print('Spark version:', spark.version)
print('Master:', sc.master)
print('Default parallelism:', sc.defaultParallelism)
print('Executors (from getExecutorMemoryStatus, if available):')
try:
    print(sc._jsc.sc().getExecutorMemoryStatus().keySet())
except Exception as e:
    print('  (could not introspect executors directly:', e, ')')

## 1. RDD — basic transformations & actions

In [ ]:
rdd = sc.parallelize(range(1, 100001), numSlices=64)
print('count:', rdd.count())
print('first:', rdd.first())
print('take(5):', rdd.take(5))
mapped = rdd.map(lambda x: x * 2)
filtered = mapped.filter(lambda x: x % 3 == 0)
print('map+filter count:', filtered.count())
flat = sc.parallelize([[1,2,3],[4,5],[6]]).flatMap(lambda xs: xs)
print('flatMap collect:', flat.collect())

In [ ]:
# distinct / union / intersection / subtract
a = sc.parallelize([1,2,2,3,4,4,5])
b = sc.parallelize([3,4,5,6,7])
print('distinct:', sorted(a.distinct().collect()))
print('union:', sorted(a.union(b).collect()))
print('intersection:', sorted(a.intersection(b).collect()))
print('subtract:', sorted(a.subtract(b).collect()))

In [ ]:
# reduce / fold / aggregate
nums = sc.parallelize(range(1, 1001), numSlices=32)
print('reduce sum:', nums.reduce(lambda x, y: x + y))
print('fold sum:', nums.fold(0, lambda x, y: x + y))
seqOp = lambda acc, v: (acc[0] + v, acc[1] + 1)
combOp = lambda a, b: (a[0] + b[0], a[1] + b[1])
total, cnt = nums.aggregate((0, 0), seqOp, combOp)
print('aggregate (sum, count):', total, cnt)

In [ ]:
# mapPartitions / mapPartitionsWithIndex — confirms real multi-host execution
import socket
def partition_host(idx, it):
    host = socket.gethostname()
    n = sum(1 for _ in it)
    yield (host, idx, n)
wide = sc.parallelize(range(200), numSlices=200)
hosts = sorted(wide.mapPartitionsWithIndex(partition_host).collect())
for h, idx, n in hosts[:10]:
    print(f'partition {idx:3d} -> {h:20s} ({n} records)')
distinct_hosts = set(h for h, _, _ in hosts)
print('DISTINCT EXECUTOR HOSTS SEEN:', distinct_hosts)

## 2. RDD — pair RDD / shuffle operations

In [ ]:
pairs = sc.parallelize([(i % 50, 1) for i in range(1_000_000)], numSlices=100)
counts = dict(pairs.reduceByKey(lambda a, b: a + b).collect())
print('reduceByKey key 0:', counts[0])
grouped = pairs.groupByKey().mapValues(len).collect()
print('groupByKey sample:', sorted(grouped)[:5])
sorted_keys = pairs.sortByKey().keys().distinct().collect()
print('sortByKey distinct keys (first 10):', sorted(sorted_keys)[:10])

In [ ]:
# aggregateByKey / combineByKey
kv = sc.parallelize([(i % 10, i) for i in range(10000)], numSlices=32)
aggd = kv.aggregateByKey((0, 0), lambda acc, v: (acc[0]+v, acc[1]+1), lambda a, b: (a[0]+b[0], a[1]+b[1])).collectAsMap()
print('aggregateByKey key 0 (sum, count):', aggd[0])

def create_combiner(v): return [v]
def merge_value(acc, v): acc.append(v); return acc
def merge_combiners(a, b): return a + b
combined = kv.combineByKey(create_combiner, merge_value, merge_combiners).mapValues(len).collectAsMap()
print('combineByKey key 0 count:', combined[0])

In [ ]:
# join / leftOuterJoin / rightOuterJoin / fullOuterJoin / cogroup
left = sc.parallelize([(1,'a'), (2,'b'), (3,'c')])
right = sc.parallelize([(2,'x'), (3,'y'), (4,'z')])
print('join:', sorted(left.join(right).collect()))
print('leftOuterJoin:', sorted(left.leftOuterJoin(right).collect()))
print('rightOuterJoin:', sorted(left.rightOuterJoin(right).collect()))
print('fullOuterJoin:', sorted(left.fullOuterJoin(right).collect()))
cogrouped = left.cogroup(right).mapValues(lambda t: (list(t[0]), list(t[1]))).collect()
print('cogroup:', sorted(cogrouped))

In [ ]:
# cartesian (expensive — keep small)
x = sc.parallelize([1,2,3])
y = sc.parallelize(['a','b'])
print('cartesian:', sorted(x.cartesian(y).collect()))

# zip / zipWithIndex
z1 = sc.parallelize(range(5), numSlices=1)
z2 = sc.parallelize(['a','b','c','d','e'], numSlices=1)
print('zip:', z1.zip(z2).collect())
print('zipWithIndex:', sc.parallelize(['p','q','r']).zipWithIndex().collect())

## 3. RDD — broadcast variables, accumulators, persistence

In [ ]:
lookup = {i: f'label_{i}' for i in range(100)}
bcast = sc.broadcast(lookup)
resolved = sc.parallelize(range(10)).map(lambda i: bcast.value[i]).collect()
print('broadcast result:', resolved)

acc = sc.accumulator(0)
def count_evens(x):
    global acc
    if x % 2 == 0:
        acc.add(1)
sc.parallelize(range(1000)).foreach(count_evens)
print('accumulator (even count):', acc.value)

In [ ]:
# persist / cache / unpersist, repartition / coalesce
cached = sc.parallelize(range(100000), numSlices=50).map(lambda x: x * x)
cached.persist()
print('first pass count:', cached.count())
print('second pass sum (from cache):', cached.sum())
cached.unpersist()
print('repartition to 10, new partition count:', cached.repartition(10).getNumPartitions())
print('coalesce to 4, new partition count:', cached.coalesce(4).getNumPartitions())

## 4. DataFrame — core transformations

In [ ]:
from pyspark.sql import functions as F, Row

df = spark.range(0, 1_000_000).withColumn('key', F.col('id') % 1000).repartition(64)
df.select('id', 'key').filter(F.col('key') < 5).show(5)
df2 = df.withColumn('key_sq', F.col('key') ** 2).withColumn('key_str', F.col('key').cast('string'))
df2.printSchema()

In [ ]:
# groupBy / agg / orderBy / distinct / dropDuplicates
agg = df.groupBy('key').agg(F.count('*').alias('cnt'), F.avg('id').alias('avg_id'), F.max('id').alias('max_id'))
agg.orderBy(F.desc('cnt')).show(5)
print('distinct keys:', df.select('key').distinct().count())
print('dropDuplicates on key:', df.dropDuplicates(['key']).count())

In [ ]:
# joins: inner / left / right / outer / cross / broadcast hint
small = spark.range(0, 1000).withColumnRenamed('id', 'key').withColumn('label', F.concat(F.lit('grp_'), F.col('key')))
print('inner join:', df.join(small, 'key', 'inner').count())
print('left join:', df.join(small, 'key', 'left').count())
print('right join (small right):', small.join(df, 'key', 'right').count())
print('full outer join:', df.join(small, 'key', 'outer').count())
print('broadcast join count:', df.join(F.broadcast(small), 'key', 'inner').count())

In [ ]:
# window functions
from pyspark.sql.window import Window
w = Window.partitionBy('key').orderBy(F.desc('id'))
ranked = df.withColumn('rn', F.row_number().over(w)).withColumn('rank', F.rank().over(w))
ranked.filter(F.col('rn') == 1).show(5)

w_running = Window.partitionBy('key').orderBy('id').rowsBetween(Window.unboundedPreceding, Window.currentRow)
running = df.withColumn('running_sum', F.sum('id').over(w_running))
running.show(5)

In [ ]:
# pivot / explode
sales = spark.createDataFrame([
    ('north', 'q1', 100), ('north', 'q2', 150), ('south', 'q1', 200), ('south', 'q2', 120)
], ['region', 'quarter', 'amount'])
sales.groupBy('region').pivot('quarter').sum('amount').show()

arr_df = spark.createDataFrame([(1, [10,20,30]), (2, [40,50])], ['id', 'vals'])
arr_df.select('id', F.explode('vals').alias('val')).show()

In [ ]:
# na handling
nulls = spark.createDataFrame([(1, None), (2, 'b'), (None, 'c')], ['id', 'val'])
nulls.na.drop().show()
nulls.na.fill({'val': 'missing', 'id': -1}).show()

## 5. DataFrame — UDFs (Python closures shipped to executors)

In [ ]:
from pyspark.sql.types import StringType

def classify(k):
    if k < 250: return 'low'
    elif k < 750: return 'mid'
    else: return 'high'

classify_udf = F.udf(classify, StringType())
df.withColumn('bucket', classify_udf('key')).groupBy('bucket').count().show()

In [ ]:
# pandas_udf (vectorized, Arrow-based)
import pandas as pd
from pyspark.sql.functions import pandas_udf

@pandas_udf('double')
def key_times_pi(s: pd.Series) -> pd.Series:
    return s * 3.14159

df.withColumn('key_pi', key_times_pi('key')).show(5)

## 6. spark.sql() string queries + temp views

In [ ]:
df.createOrReplaceTempView('nums')
spark.sql('SELECT key, COUNT(*) as cnt FROM nums GROUP BY key ORDER BY cnt DESC LIMIT 5').show()

## 7. Read/write round trip (parquet)

In [ ]:
import os
out_path = os.environ.get('SPARK_WAREHOUSE_DIR', '/tmp/spark-warehouse') + '/pyspark_test_roundtrip'
df.select('id', 'key').write.mode('overwrite').partitionBy('key').parquet(out_path)
read_back = spark.read.parquet(out_path)
print('round trip row count matches:', read_back.count() == df.count())

## 8. RDD <-> DataFrame interop

In [ ]:
rdd_from_df = df.rdd.map(lambda row: (row['key'], row['id'] * 2))
print('rdd_from_df sample:', rdd_from_df.take(5))

df_from_rdd = spark.createDataFrame(sc.parallelize([Row(x=i, y=i*i) for i in range(10)]))
df_from_rdd.show()